In [3]:
from the_well.data.datasets import WellDataset
from pathlib import Path
import sys
import os

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.mini_well import create_mini_well

In [7]:
# Cell 2 — Inspect original dataset
well_base_path    = '/Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets'
#well_dataset_name = 'turbulent_radiative_layer_2D'
#well_dataset_name = 'planetswe'
well_dataset_name = 'rayleigh_benard'

MINI_DATASET  = f'{well_dataset_name}_mini'

for split in ['train', 'valid', 'test']:
    ds = WellDataset(
        well_base_path=well_base_path,
        well_dataset_name=well_dataset_name,
        well_split_name=split,
    )
    print(f"\n── {split} ──")
    print(f"  Spatial resolution   : {ds.size_tuple}")
    print(f"  N files              : {ds.n_files}")
    print(f"  Trajectories/file    : {ds.n_trajectories_per_file}")
    print(f"  Steps/traj           : {ds.n_steps_per_trajectory}")
    print(f"  Total trajectories   : {sum(ds.n_trajectories_per_file)}")


── train ──
  Spatial resolution   : (512, 128)
  N files              : 1
  Trajectories/file    : [40]
  Steps/traj           : [200]
  Total trajectories   : 40

── valid ──
  Spatial resolution   : (512, 128)
  N files              : 1
  Trajectories/file    : [40]
  Steps/traj           : [200]
  Total trajectories   : 40

── test ──
  Spatial resolution   : (512, 128)
  N files              : 1
  Trajectories/file    : [40]
  Steps/traj           : [200]
  Total trajectories   : 40


In [8]:
# Cell 3 — Mini-well parameters
# ── Spatial ──────────────────────────────────────────────────
# Keep 512x128 for now → factor=1
SPATIAL_FACTOR = 1

# ── Time ─────────────────────────────────────────────────────
# Original: 200 steps
# time_fraction=1.0  → use all 200 steps
# time_downsample_factor=2 → keep every 2nd → 100 steps final
TIME_FACTOR    = 2
TIME_FRACTION  = 1.0

# ── Trajectories per file ────────────────────────────────────
# Train: 10/file × 34 files = 340 trajectories (covers all Ra/Pr regimes)
# Val/test: take more since fewer files exist → better coverage
MAX_TRAJ_PER_FILE = {'train': 10, 'valid': 20, 'test': 20}

#MINI_BASE = f'{well_base_path}/{well_dataset_name}/mini_well_data'
#os.makedirs(MINI_BASE, exist_ok=True)

In [9]:
for split, max_traj in MAX_TRAJ_PER_FILE.items():
    ds_full = WellDataset(
        well_base_path=well_base_path,
        well_dataset_name=well_dataset_name,
        well_split_name=split,
    )
    print(f"\n{'='*50}  {split}  {'='*50}")

    for file_path in ds_full.files_paths:
        ds_single = WellDataset(
            well_base_path=well_base_path,
            well_dataset_name=well_dataset_name,
            well_split_name=split,
            include_filters=[Path(file_path).name],
        )
        create_mini_well(
            dataset=ds_single,
            well_base_path=well_base_path,
            mini_dataset_name=MINI_DATASET,
            spatial_downsample_factor=SPATIAL_FACTOR,
            time_downsample_factor=TIME_FACTOR,
            max_trajectories=max_traj,
            split=split,
            time_fraction=TIME_FRACTION,
        )

print("\nAll splits done ✓")


==================================================  train  ==================================================
  Copied stats.yaml → /Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets/rayleigh_benard_mini/stats.yaml


train: 100%|██████████| 1/1 [00:03<00:00,  3.42s/it]


    rayleigh_benard_Rayleigh_1e6_Prandtl_2.hdf5: 10 traj written
  ✓ train: 10 trajectories → /Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets/rayleigh_benard_mini/data/train

==================================================  valid  ==================================================


valid: 100%|██████████| 1/1 [00:03<00:00,  3.35s/it]


    rayleigh_benard_Rayleigh_1e6_Prandtl_2.hdf5: 20 traj written
  ✓ valid: 20 trajectories → /Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets/rayleigh_benard_mini/data/valid

==================================================  test  ==================================================


test: 100%|██████████| 1/1 [00:03<00:00,  3.25s/it]

    rayleigh_benard_Rayleigh_1e6_Prandtl_2.hdf5: 20 traj written
  ✓ test: 20 trajectories → /Users/gabrieleaccarino/Workspace/multi-scale-wavelet-flow-matching/data/datasets/rayleigh_benard_mini/data/test

All splits done ✓


In [10]:
# Cell 5 — Verify the mini dataset
for split in ['train', 'valid', 'test']:
    ds_mini = WellDataset(
        path=os.path.join(well_base_path, MINI_DATASET, 'data', split), 
        #well_base_path=well_base_path,
        #well_dataset_name=MINI_DATASET,
        #well_split_name=split,
    )
    total_traj = sum(ds_mini.n_trajectories_per_file)
    print(f"\n── mini/{split} ──")
    print(f"  Spatial resolution : {ds_mini.size_tuple}")
    print(f"  N files            : {ds_mini.n_files}")
    print(f"  Trajectories/file  : {ds_mini.n_trajectories_per_file}")
    print(f"  Steps/traj         : {ds_mini.n_steps_per_trajectory}")
    print(f"  Total trajectories : {total_traj}")
    print(f"  Total windows      : {len(ds_mini)}")


── mini/train ──
  Spatial resolution : (512, 128)
  N files            : 1
  Trajectories/file  : [10]
  Steps/traj         : [100]
  Total trajectories : 10
  Total windows      : 990

── mini/valid ──
  Spatial resolution : (512, 128)
  N files            : 1
  Trajectories/file  : [20]
  Steps/traj         : [100]
  Total trajectories : 20
  Total windows      : 1980

── mini/test ──
  Spatial resolution : (512, 128)
  N files            : 1
  Trajectories/file  : [20]
  Steps/traj         : [100]
  Total trajectories : 20
  Total windows      : 1980
